# Thread workers

> Threaded workers to isolate IO operations and CPU intensive tasks into a background thread

In [ ]:
#| default_exp threadworkers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import threading
import queue
import time

from typing import NamedTuple

import logging


In [ ]:
#| exporti
syslog = logging.getLogger("root." + __name__)


In [ ]:
def fun(arg1, kwarg1='default', **kwargs):
    print(f"arg1: {arg1}, kwarg1: {kwarg1}, other kwargs: {kwargs}")

fun(1, kwarg1='changed', extra1='extra1', extra2='extra2')
fun(1, extra1='extra1', extra2='extra2')

arg1: 1, kwarg1: changed, other kwargs: {'extra1': 'extra1', 'extra2': 'extra2'}
arg1: 1, kwarg1: default, other kwargs: {'extra1': 'extra1', 'extra2': 'extra2'}


In [ ]:
#| export
class WorkerArgs(NamedTuple):
    args: tuple
    kwargs: dict


In [ ]:
#| export
def bind_task_queue_put(
        task_queue,     # The queue to put work items into
        **put_kwargs    # Additional kwargs for queue.put()
) -> callable:
    
    "Binds a queue with kwargs for putting work items into the queue"

    def put_task_item(*work_args, **work_kwargs):
        task_queue.put(item=WorkerArgs(work_args, work_kwargs), **put_kwargs)

    return put_task_item


In [ ]:
#| export
def bind_task_queue_get(
        product_queue,      # The queue to get produced items from
        **get_kwargs        # Additional kwargs for queue.get()
) -> callable:
    
    "Binds a queue and kwargs for getting produced items from the queue"

    def get_task_item():
        try:
            item = product_queue.get(**get_kwargs)

        except queue.Empty:
            item = None

        return item

    return get_task_item

In [ ]:
test_queue = queue.Queue(maxsize=10)

get_item = bind_task_queue_get(test_queue, block=False)


In [ ]:
for i in range(5):
    test_queue.put(i)


In [ ]:
get_item()



In [ ]:
#| export

def bind_task_queue_generator(
        product_queue       # The queue to get produced items from
) -> callable:
    
    "Binds a queue and kwargs for getting produced items from the queue"

    def get_task_items():
        try:
            while True:
                item = product_queue.get_nowait()
                yield item

        except queue.Empty:
            pass

    return get_task_items

In [ ]:
test_queue = queue.Queue(maxsize=10)

get_items = bind_task_queue_generator(test_queue)


In [ ]:
for i in range(5):
    test_queue.put(i)


In [ ]:
list(get_items())



[0, 1, 2, 3, 4]

In [ ]:
#| exporti

def workloop(
        task_queue: queue.Queue,
        worker_fn:  callable,
        product_queue: queue.Queue,
        stop_event: threading.Event,
    ):
        """
        A worker thread loop that gets items from a queue, processes 
        them with a worker function and puts it's result on a queue.
        """

        sleepfor = 0.05 # initial sleep time when no work is available
        sleepmax = 0.5  # max sleep time when no work is available

        # indefinately keep getting new items from the queue to process
        while True:
            
            try:
                item = None
                try:
                    # by default queue.get() waits for new items
                    item = task_queue.get_nowait()
                    task_queue.task_done()
                    sleepfor = 0.05

                except queue.Empty:
                    if stop_event.is_set():
                        break
                    else:
                        time.sleep(sleepfor)
                        sleepfor = min(sleepfor * 1.2, sleepmax)

                try:
                    result = worker_fn(item)
                    product_queue.put_nowait(result)

                except queue.Full:
                    pass


            except Exception as x:
                syslog.exception("Exception: %s", x, exc_info=True, stack_info=True)



In [ ]:
import nbdev; nbdev.nbdev_export()